In [1]:
# Install google-cloud-bigquery if not already available
!pip install google-cloud-bigquery pandas db-dtypes --quiet

In [2]:
PROJECT_ID = "qwiklabs-gcp-00-871084f9eb9e"
DATASET_ID = "fraud_detection"
RAW_TABLE  = "fraud_data_raw"
TRAIN_TABLE = "fraud_training_data"
LOCATION   = "US"
GCS_URI    = "gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv"

In [3]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
dataset_ref.default_table_expiration_ms = 14 * 24 * 60 * 60 * 1000  # 14 days

dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset '{DATASET_ID}' ready.")

Dataset 'fraud_detection' ready.


In [4]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,       # skip header row
    autodetect=True,
    max_bad_records=10,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

table_ref = f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}"
load_job = client.load_table_from_uri(GCS_URI, table_ref, job_config=job_config)
load_job.result()  # wait for completion

table = client.get_table(table_ref)
print(f"Loaded {table.num_rows} rows into {table_ref}")

Loaded 50000 rows into qwiklabs-gcp-00-871084f9eb9e.fraud_detection.fraud_data_raw


In [5]:
query = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}` LIMIT 5"
df_sample = client.query(query).to_dataframe()
print(df_sample.dtypes)
df_sample.head()

Applicant_ID                         Int64
Age                                  Int64
Employment_Status                   object
Income                               Int64
Number_of_Dependents                 Int64
Amount_Requested                     Int64
Previous_Assistance_Received       boolean
Previous_Assistance_Date            dbdate
Supporting_Doc_Verified            boolean
Application_Frequency_Last_Year      Int64
IP_Address                          object
Device_Type                         object
Application_Date                    dbdate
Fraudulent                           Int64
dtype: object


,Applicant_ID,Age,Employment_Status,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Device_Type,Application_Date,Fraudulent
0,217,65,Unemployed,28984,4,5872,False,NaT,False,1,156.133.45.45,Mobile,2024-08-18,0
1,226,54,Self-Employed,0,1,6631,False,NaT,False,1,245.13.80.245,Tablet,2024-05-11,0
2,240,26,Self-Employed,64477,5,8612,False,NaT,True,1,213.103.170.95,Mobile,2024-08-14,0
3,252,28,Unemployed,28576,4,2951,False,NaT,True,1,234.179.149.207,Desktop,2024-06-12,0
4,266,43,Employed,44930,5,2324,False,NaT,False,1,66.109.96.227,Mobile,2024-08-16,0


In [7]:
feature_engineering_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}` AS

WITH age_binned AS (
  SELECT *,
    CASE
      WHEN Age BETWEEN 18 AND 24 THEN '18-24'
      WHEN Age BETWEEN 25 AND 34 THEN '25-34'
      WHEN Age BETWEEN 35 AND 44 THEN '35-44'
      WHEN Age BETWEEN 45 AND 54 THEN '45-54'
      WHEN Age BETWEEN 55 AND 64 THEN '55-64'
      WHEN Age >= 65             THEN '65+'
      ELSE 'Unknown'
    END AS Age_Bin
  FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
)

SELECT
  -- One-hot encode: Employment_Status (STRING)
  IF(Employment_Status = 'Employed',      1, 0) AS Employed,
  IF(Employment_Status = 'Unemployed',    1, 0) AS Unemployed,
  IF(Employment_Status = 'Self-Employed', 1, 0) AS Self_Employed,
  IF(Employment_Status = 'Retired',       1, 0) AS Retired,
  IF(Employment_Status = 'Student',       1, 0) AS Student,

  -- One-hot encode: Device_Type (STRING)
  IF(Device_Type = 'Mobile',  1, 0) AS Device_Mobile,
  IF(Device_Type = 'Desktop', 1, 0) AS Device_Desktop,
  IF(Device_Type = 'Tablet',  1, 0) AS Device_Tablet,

  -- One-hot encode: Age bins (INT64)
  IF(Age_Bin = '18-24', 1, 0) AS Age_18_24,
  IF(Age_Bin = '25-34', 1, 0) AS Age_25_34,
  IF(Age_Bin = '35-44', 1, 0) AS Age_35_44,
  IF(Age_Bin = '45-54', 1, 0) AS Age_45_54,
  IF(Age_Bin = '55-64', 1, 0) AS Age_55_64,
  IF(Age_Bin = '65+',   1, 0) AS Age_65_Plus,

  -- Income to Amount Requested ratio (INT64 / INT64 → FLOAT64)
  SAFE_DIVIDE(Income, Amount_Requested) AS Income_to_Amount_Requested,

  -- Time Since Previous Assistance (DATE type → DAY diff; 0 if NULL)
  IF(
    Previous_Assistance_Received = TRUE AND Previous_Assistance_Date IS NOT NULL,
    DATE_DIFF(CURRENT_DATE(), Previous_Assistance_Date, DAY),
    0
  ) AS Time_Since_Previous_Assistance,

  -- Boolean → 0/1 (BOOL type: compare directly with TRUE)
  IF(Previous_Assistance_Received = TRUE, 1, 0) AS Previous_Assistance_Received,
  IF(Supporting_Doc_Verified = TRUE,      1, 0) AS Supporting_Doc_Verified,

  -- Keep remaining columns as-is
  Applicant_ID,
  Age,
  Income,
  Number_of_Dependents,
  Amount_Requested,
  Application_Frequency_Last_Year,
  IP_Address,
  Application_Date,
  Fraudulent

FROM age_binned;
"""

job = client.query(feature_engineering_sql)
job.result()
print("✅ Feature engineering complete. Table 'fraud_training_data' created.")

✅ Feature engineering complete. Table 'fraud_training_data' created.


In [8]:
verify_query = f"""
  SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}` LIMIT 5
"""
df_result = client.query(verify_query).to_dataframe()
print(f"Output table shape: {df_result.shape}")
df_result.head()

Output table shape: (5, 27)


,Employed,Unemployed,Self_Employed,Retired,Student,Device_Mobile,Device_Desktop,Device_Tablet,Age_18_24,Age_25_34,...,Supporting_Doc_Verified,Applicant_ID,Age,Income,Number_of_Dependents,Amount_Requested,Application_Frequency_Last_Year,IP_Address,Application_Date,Fraudulent
0,1,0,0,0,0,1,0,0,1,0,...,1,30494,18,70041,5,3016,2,253.215.114.193,2024-06-25,0
1,1,0,0,0,0,1,0,0,1,0,...,1,35361,18,26404,2,2759,2,228.57.68.63,2024-04-15,0
2,0,1,0,0,0,0,0,1,1,0,...,1,42193,18,0,3,7972,2,131.190.192.43,2024-02-06,0
3,0,1,0,0,0,0,0,1,1,0,...,1,45153,18,37156,0,3292,2,156.133.24.175,2024-08-22,0
4,0,0,1,0,0,1,0,0,1,0,...,0,47118,18,56582,2,2367,4,80.46.141.147,2024-10-13,0


In [9]:
count_query = f"SELECT COUNT(*) AS total FROM `{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}`"
df_count = client.query(count_query).to_dataframe()
print(f"Total rows in fraud_training_data: {df_count['total'][0]}")

Total rows in fraud_training_data: 50000
